# 제주 특산물 가격 예측 - EMA 래그 피처 추가 v4.1.0

| 항목 | 내용 |
|------|------|
| **버전** | v4.1.0 |
| **날짜** | 2026-03-16 |
| **모델** | CatBoost + LightGBM 앙상블 |
| **검증** | 2022년 3월 (테스트와 동일 계절) |
| **전처리** | v1.0.1 + 사이클 피처 + **EMA 래그 피처 (7/30/60/120일)** |
| **CPU** | i7-1360P 16스레드 × 80% = 12스레드 |
| **출력** | results/submission_v4.1.0.csv |

## v4.1.0 변경 내용 (v4.0.0 대비)
| 항목 | v4.0.0 | v4.1.0 |
|------|--------|--------|
| 래그 피처 | 없음 | **EMA 7/30/60/120일 추가** (김대원님 제안 수용) |
| EMA 계산 기준 | — | **품목별 일평균 가격** (동일 날짜 다중 행 누수 방지) |
| 데이터 누수 방지 | — | **shift(1)** + ignore_na=True |
| 피처 수 | 19개 | **23개** (EMA 4개 추가) |
| 나머지 | v4.0.0 동일 | v4.0.0 동일 |

---
## 버전 히스토리 & DACON 제출 결과 분석

### 버전별 성능 요약

| 버전 | 모델 | 핵심 변경 | 내부 검증 MAE | DACON Public | DACON Private |
|------|------|----------|--------------:|-------------:|--------------:|
| **v1.0.0** | DNN | 기본 DNN, 전처리 미흡 | — | 1695.6 | 1721.4 |
| **v1.0.1** | DNN | DACON 1위 전처리(공휴일·누적 week_num·TG sqrt·후처리) + 기본 Dense | 522 원/kg | **658.6** ✅ | **825.0** ✅ |
| **v1.1.0** | DNN+Emb | item·corporation·location을 Embedding 레이어로 처리 | 537 원/kg | 미제출 | 미제출 |
| **v1.2.0** | DNN+Emb+Res | Embedding + Residual Block(skip connection) 추가 | 532 원/kg | 1155.3 ❌ | 1326.3 ❌ |
| **v2.0.0** | LSTM | 14일 슬라이딩 윈도우 + 오토레그레시브 예측 | 890 원/kg | 783.9 | 1030.2 |
| **v3.0.0** | DNN+CB+XGB | DNN·CatBoost·XGBoost 3모델 앙상블 도입 | ~510 원/kg | 미제출 | 미제출 |
| **v3.1.0** | DNN(Res)+CB+XGB | DNN을 Residual 구조(v1.2.0)로 교체한 앙상블 | ~510 원/kg | 910.9 ❌ | 1062.6 ❌ |
| **v4.0.0** | LGB+CB | DNN 제거, LightGBM 추가 → CPU 최적 순수 트리 앙상블 | ~510 원/kg | — | — |
| **v4.1.0** | LGB+CB | **EMA 래그 피처 추가** (7/30/60/120일, 품목별 일평균 기반) | — | — | — |

---

### v4.1.0 EMA 피처 도입 근거

- **김대원님 분석**: 품목별 가격 시계열에 명확한 추세/계절성 존재 → EMA가 이 추세를 직접 인코딩
- **SMA 대비 EMA 장점**: 일요일 등 거래 없는 날(price=0)이 존재할 때 EMA가 더 빠르게 회복
- **shift(1)**: 당일 가격이 자기 EMA 계산에 참여하지 않도록 방지 (데이터 누수 차단)
- **일평균 기준**: 동일 날짜·품목에 복수 행(법인·지역 조합) 존재 → 행 단위 shift 시 같은 날 행 간 누수 발생 → 일평균 집계 후 EMA 계산

---
## 1. 라이브러리 로드

In [ ]:
try:
    import lightgbm as lgb
    print(f'LightGBM: {lgb.__version__}')
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'lightgbm'], check=True)
    import lightgbm as lgb
    print(f'LightGBM installed: {lgb.__version__}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, datetime, random, os, platform
warnings.filterwarnings('ignore')

import holidays
import lightgbm as lgb
from catboost import CatBoostRegressor

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

SEED = 2024
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

N_CPU_TOTAL = os.cpu_count()
N_JOBS = max(1, int(N_CPU_TOTAL * 0.8))
print(f'총 CPU 스레드: {N_CPU_TOTAL}  →  사용 스레드: {N_JOBS} (80%)')
print(f'LightGBM: {lgb.__version__} | CatBoost: {__import__("catboost").__version__}')

---
## 2. 데이터 로드

In [ ]:
DATA_PATH = '../data/'
train = pd.read_csv(DATA_PATH + 'train.csv', encoding='utf-8-sig')
test  = pd.read_csv(DATA_PATH + 'test.csv',  encoding='utf-8-sig')
sub   = pd.read_csv(DATA_PATH + 'sample_submission.csv', encoding='utf-8-sig')
print(f'train: {train.shape}, test: {test.shape}')

---
## 3. 전처리 + 이상치 + 타겟 변환 + EMA 래그 피처 (v4.1.0 신규)

### EMA 계산 방식
- **품목별 일평균 가격** 기준: 동일 날짜·품목에 복수 행이 존재하므로 행 단위 shift 시 같은 날 내 누수 발생 → 일평균 집계 후 EMA 계산
- **shift(1)**: 당일 가격이 자신의 EMA에 포함되지 않도록 1일 후행
- **ignore_na=True**: 일요일·공휴일 등 거래 없는 날(price=NaN 또는 0)을 건너뛰고 EMA 유지
- **테스트 데이터**: 가격 미지 → 마지막 train EMA 값이 freeze되어 전파 (최선의 추정값)

In [ ]:
EMA_SPANS = [7, 30, 60, 120]
EMA_COLS  = [f'price_{w}d_ema' for w in EMA_SPANS]

def add_features(df):
    """사이클 인코딩 + 추가 시간 피처"""
    df['quarter']      = df['timestamp'].dt.quarter
    df['is_weekend']   = (df['week_day'] >= 5).astype(int)
    df['day_of_year']  = df['timestamp'].dt.dayofyear

    df['month_sin']       = np.sin(2 * np.pi * df['month']       / 12)
    df['month_cos']       = np.cos(2 * np.pi * df['month']       / 12)
    df['week_day_sin']    = np.sin(2 * np.pi * df['week_day']    / 7)
    df['week_day_cos']    = np.cos(2 * np.pi * df['week_day']    / 7)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)
    return df

def add_ema_features(df):
    """
    품목별 일평균 가격 기준 EMA 피처 생성
    - 데이터 누수 방지: shift(1) 적용 (당일 가격 제외)
    - 다중 행 누수 방지: 행 단위가 아닌 일평균 기준으로 계산
    - 테스트 기간: ignore_na=True로 마지막 train 추세값 유지
    """
    # 품목·날짜별 일평균 가격 집계 (train: 실제값, test: NaN)
    daily_avg = (
        df.sort_values(['item', 'timestamp'])
          .groupby(['item', 'timestamp'])['price']
          .mean()
          .rename('price_avg')
          .reset_index()
    )

    # 품목별로 shift(1) → EMA 계산
    for w in EMA_SPANS:
        col = f'price_{w}d_ema'
        daily_avg[col] = daily_avg.groupby('item')['price_avg'].transform(
            lambda x: x.shift(1).ewm(span=w, min_periods=1, ignore_na=True).mean()
        )

    # 원본 df에 merge (item + timestamp 키)
    df = df.merge(daily_avg[['item', 'timestamp'] + EMA_COLS],
                  on=['item', 'timestamp'], how='left')
    return df

def pre_all(train, test):
    train['timestamp'] = pd.to_datetime(train['timestamp'])
    test['timestamp']  = pd.to_datetime(test['timestamp'])
    df = pd.concat([train, test]).reset_index(drop=True)
    df.rename(columns={'supply(kg)': 'supply', 'price(원/kg)': 'price'}, inplace=True)

    df['year']     = df['timestamp'].dt.year
    df['month']    = df['timestamp'].dt.month
    df['day']      = df['timestamp'].dt.day
    df['week_day'] = df['timestamp'].dt.weekday

    le = LabelEncoder()
    df['year_month'] = df['timestamp'].map(lambda x: f'{x.year}-{x.month}')
    df['year_month'] = le.fit_transform(df['year_month'])

    df['week'] = df['timestamp'].map(
        lambda x: datetime.datetime(x.year, x.month, x.day).isocalendar()[1]
    )
    week_offsets = {2019: 0, 2020: 52, 2021: 52+53, 2022: 52+53+53, 2023: 52+53+53+52}
    df['week_num'] = df.apply(lambda r: int(r['week']) + week_offsets.get(r['year'], 0), axis=1)
    df.loc[df['timestamp'] == '2019-12-30', 'week_num'] = 52
    df.loc[df['timestamp'] == '2019-12-31', 'week_num'] = 52

    kr_holi = holidays.KR()
    df['holiday'] = df['timestamp'].map(lambda x: 1 if x in kr_holi else 0)

    df = add_features(df)

    # ── v4.1.0 신규: EMA 래그 피처 ──
    df = add_ema_features(df)

    train_out = df[~df['price'].isnull()].sort_values('timestamp').reset_index(drop=True)
    test_out  = df[ df['price'].isnull()].sort_values('timestamp').reset_index(drop=True)
    print(f'전처리 완료 — train: {train_out.shape}, test: {test_out.shape}')
    print(f'추가된 EMA 피처: {EMA_COLS}')
    return train_out, test_out

train_pre, test_pre = pre_all(train, test)

# 이상치 처리
outlier_thresholds = {'TG': 20000, 'RD': 5000, 'BC': 8000, 'CB': 2300}
for item, thr in outlier_thresholds.items():
    idx = train_pre[(train_pre['item'] == item) & (train_pre['price'] > thr)].index
    if len(idx):
        mean_p = train_pre[(train_pre['item'] == item) & (train_pre['price'] != 0)]['price'].mean()
        train_pre.loc[idx, 'price'] = mean_p
        print(f'{item}: {len(idx)}개 이상치 → 평균({mean_p:.0f})')

# 타겟 변환
train_pre['price_transformed'] = np.where(
    train_pre['item'] == 'TG',
    np.sqrt(train_pre['price']),
    np.log1p(train_pre['price'])
)

# TG 공휴일 보정
tg_mask = (train_pre['item'] == 'TG') & (train_pre['holiday'] == 1) & (train_pre['price'] != 0)
active_holi = list(train_pre[tg_mask].groupby('timestamp').count().reset_index()['timestamp'])
fix_idx = train_pre[train_pre['timestamp'].isin(active_holi)].index
train_pre.loc[fix_idx, 'holiday'] = 0
print(f'TG 공휴일 보정: {len(fix_idx)}개')

In [ ]:
# EMA 피처 통계 확인
print('=== EMA 피처 통계 (train) ===')
print(train_pre[EMA_COLS].describe().round(1))
print(f'\nNaN 개수: {train_pre[EMA_COLS].isnull().sum().to_dict()}')
print('(NaN은 각 품목 첫 행에만 발생 — 모델이 자동 처리)')

print('\n=== EMA 피처 통계 (test) ===')
print(test_pre[EMA_COLS].describe().round(1))
print(f'NaN 개수: {test_pre[EMA_COLS].isnull().sum().to_dict()}')

In [ ]:
# EMA 피처 시각화 (품목별 샘플)
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
items = ['TG', 'CR', 'CB', 'RD', 'BC']
colors = ['steelblue', 'tomato', 'seagreen', 'darkorange', 'mediumpurple']

for ax, item, color in zip(axes.flat, items, colors):
    mask = train_pre['item'] == item
    sample = train_pre[mask].drop_duplicates('timestamp').sort_values('timestamp')
    ax.plot(sample['timestamp'], sample['price'],      alpha=0.3, lw=1,   color=color,      label='실제가격')
    ax.plot(sample['timestamp'], sample['price_7d_ema'],  alpha=0.8, lw=1.2, color='crimson',  label='EMA-7')
    ax.plot(sample['timestamp'], sample['price_30d_ema'], alpha=0.8, lw=1.5, color='navy',     label='EMA-30')
    ax.plot(sample['timestamp'], sample['price_60d_ema'], alpha=0.8, lw=1.5, color='darkgreen',label='EMA-60')
    ax.plot(sample['timestamp'], sample['price_120d_ema'],alpha=0.8, lw=2,   color='black',    label='EMA-120')
    ax.set_title(f'{item} EMA 피처', fontsize=12)
    ax.set_xlabel('날짜'); ax.set_ylabel('가격 (원/kg)')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(alpha=0.3)

axes.flat[-1].set_visible(False)
plt.suptitle('v4.1.0 EMA 래그 피처 시각화 (품목별)', fontsize=15, fontweight='bold')
plt.tight_layout()
os.makedirs('./results', exist_ok=True)
plt.savefig('./results/ema_features_v4.1.0.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. 검증 분리 (2022년 3월) + 유틸 함수

> 테스트 데이터는 2023년 3월 → 동일 계절인 **2022년 3월**을 검증셋으로 사용

In [ ]:
TARGET_TRF = 'price_transformed'
TARGET_COL = 'price'

CAT_COLS  = ['item', 'corporation', 'location']
TIME_COLS = ['year_month', 'week_num', 'week_day', 'month', 'day', 'holiday', 'year',
             'quarter', 'is_weekend', 'day_of_year',
             'month_sin', 'month_cos', 'week_day_sin', 'week_day_cos',
             'day_of_year_sin', 'day_of_year_cos']

# v4.1.0: EMA 피처 추가
TREE_FEAT = TIME_COLS + CAT_COLS + EMA_COLS
print(f'전체 피처 수: {len(TREE_FEAT)}개')
print(f'  시간 피처: {len(TIME_COLS)}개')
print(f'  범주 피처: {len(CAT_COLS)}개')
print(f'  EMA 피처:  {len(EMA_COLS)}개  ← v4.1.0 신규')

# 2022년 3월을 검증셋으로 분리
val_mask = (train_pre['year'] == 2022) & (train_pre['month'] == 3)
train_df = train_pre[~val_mask].reset_index(drop=True)
val_df   = train_pre[ val_mask].reset_index(drop=True)

y_tr      = train_df[TARGET_TRF].values
y_vl      = val_df[TARGET_TRF].values
y_vl_orig = val_df[TARGET_COL].values
is_tg_vl  = (val_df['item'] == 'TG').values

def inverse_transform(y_trf, is_tg):
    result = np.where(
        is_tg.astype(bool),
        np.power(np.clip(y_trf, 0, None), 2),
        np.expm1(y_trf)
    )
    return np.clip(result, 0, None)

def eval_model(y_true, y_pred, name):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'  [{name}]  MAE={mae:>8,.1f}  RMSE={rmse:>8,.1f}  R²={r2:.4f}')
    return mae, rmse, r2

print(f'\n학습셋: {len(train_df):,}행  |  검증셋: {len(val_df):,}행 (2022-03)')
print(f'검증 기간: {val_df["timestamp"].min().date()} ~ {val_df["timestamp"].max().date()}')

---
## 5. 모델 1 — LightGBM

In [ ]:
X_lgb_tr   = train_df[TREE_FEAT].copy()
X_lgb_vl   = val_df[TREE_FEAT].copy()
X_lgb_test = test_pre[TREE_FEAT].copy()

# LightGBM 카테고리 피처: pandas category dtype으로 변환
for col in CAT_COLS:
    cats = pd.Categorical(
        pd.concat([X_lgb_tr[col], X_lgb_vl[col], X_lgb_test[col]])
    ).categories
    X_lgb_tr[col]   = pd.Categorical(X_lgb_tr[col],   categories=cats)
    X_lgb_vl[col]   = pd.Categorical(X_lgb_vl[col],   categories=cats)
    X_lgb_test[col] = pd.Categorical(X_lgb_test[col], categories=cats)

lgb_model = lgb.LGBMRegressor(
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=7,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=2.0,
    objective='regression_l1',
    random_state=SEED,
    n_jobs=N_JOBS,
    verbose=-1
)

lgb_model.fit(
    X_lgb_tr, y_tr,
    eval_set=[(X_lgb_vl, y_vl)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=200, verbose=False),
        lgb.log_evaluation(period=500)
    ]
)
print(f'LightGBM 최적 반복: {lgb_model.best_iteration_}')

---
## 6. 모델 2 — CatBoost

In [ ]:
X_cb_tr   = train_df[TREE_FEAT]
X_cb_vl   = val_df[TREE_FEAT]
X_cb_test = test_pre[TREE_FEAT]

cb_model = CatBoostRegressor(
    iterations=5000,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=3,
    loss_function='MAE',
    eval_metric='MAE',
    cat_features=CAT_COLS,
    random_seed=SEED,
    early_stopping_rounds=200,
    thread_count=N_JOBS,
    verbose=500
)
cb_model.fit(X_cb_tr, y_tr, eval_set=(X_cb_vl, y_vl), use_best_model=True)
print(f'CatBoost 최적 반복: {cb_model.get_best_iteration()}')

---
## 7. 개별 모델 검증 성능

In [ ]:
lgb_vl_pred_trf = lgb_model.predict(X_lgb_vl)
lgb_vl_pred     = inverse_transform(lgb_vl_pred_trf, is_tg_vl)

cb_vl_pred_trf  = cb_model.predict(X_cb_vl)
cb_vl_pred      = inverse_transform(cb_vl_pred_trf, is_tg_vl)

print('=' * 60)
print('  개별 모델 검증 성능 (2022년 3월)')
print('=' * 60)
lgb_scores = eval_model(y_vl_orig, lgb_vl_pred, 'LightGBM')
cb_scores  = eval_model(y_vl_orig, cb_vl_pred,  'CatBoost')
print('=' * 60)

---
## 8. 앙상블 (단순 평균) + 최적 모델 자동 선택

In [ ]:
ens_vl_pred = (lgb_vl_pred + cb_vl_pred) / 2

mae_lgb = mean_absolute_error(y_vl_orig, lgb_vl_pred)
mae_cb  = mean_absolute_error(y_vl_orig, cb_vl_pred)
mae_ens = mean_absolute_error(y_vl_orig, ens_vl_pred)

print(f'  LightGBM 단독  MAE: {mae_lgb:>8,.2f}')
print(f'  CatBoost 단독  MAE: {mae_cb:>8,.2f}')
print(f'  단순 평균 앙상블 MAE: {mae_ens:>8,.2f}')

best_mae  = min(mae_lgb, mae_cb, mae_ens)
if best_mae == mae_ens:
    FINAL_MODE = 'ensemble'
    final_vl_pred = ens_vl_pred
elif best_mae == mae_cb:
    FINAL_MODE = 'catboost'
    final_vl_pred = cb_vl_pred
else:
    FINAL_MODE = 'lightgbm'
    final_vl_pred = lgb_vl_pred

print(f'\n→ 최종 선택: {FINAL_MODE.upper()}  (MAE={best_mae:,.2f})')

In [ ]:
mae  = mean_absolute_error(y_vl_orig, final_vl_pred)
rmse = np.sqrt(mean_squared_error(y_vl_orig, final_vl_pred))
r2   = r2_score(y_vl_orig, final_vl_pred)

print('=' * 60)
print(f'  최종 모델({FINAL_MODE.upper()}) 검증 성능 (2022년 3월)')
print('=' * 60)
print(f'  MAE  : {mae:>10,.2f} 원/kg')
print(f'  RMSE : {rmse:>10,.2f} 원/kg')
print(f'  R²   : {r2:>10.4f}')
print('=' * 60)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = ['LightGBM', 'CatBoost', 'Ensemble(avg)']
maes   = [mae_lgb, mae_cb, mae_ens]
colors = ['steelblue', 'darkorange', 'crimson']
best_idx = maes.index(min(maes))

bars = axes[0].bar(model_names, maes, color=colors, alpha=0.85, edgecolor='white')
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)
axes[0].set_title('모델별 검증 MAE (2022년 3월)', fontsize=13)
axes[0].set_ylabel('MAE (원/kg)')
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, maes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{val:,.0f}', ha='center', va='bottom', fontsize=10)

max_v = max(y_vl_orig.max(), final_vl_pred.max())
axes[1].scatter(y_vl_orig, final_vl_pred, alpha=0.3, s=8, color=colors[best_idx])
axes[1].plot([0, max_v], [0, max_v], 'k--', lw=2)
axes[1].set_title(f'{FINAL_MODE.upper()} 실제 vs 예측  R²={r2:.4f}', fontsize=12)
axes[1].set_xlabel('실제'); axes[1].set_ylabel('예측'); axes[1].grid(alpha=0.3)

plt.suptitle('v4.1.0 검증 결과 (2022-03) — EMA 피처 포함', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('./results/ensemble_eval_v4.1.0.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. 피처 중요도 (EMA 피처 기여도 확인)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# LightGBM
lgb_imp = pd.Series(lgb_model.feature_importances_, index=TREE_FEAT).sort_values(ascending=False)
bar_colors_lgb = ['crimson' if c in EMA_COLS else 'steelblue' for c in lgb_imp.head(15).index]
lgb_imp.head(15).plot(kind='barh', ax=axes[0], color=bar_colors_lgb, alpha=0.85)
axes[0].invert_yaxis()
axes[0].set_title('LightGBM Top-15 Feature Importance\n(빨강=EMA 피처)', fontsize=12)
axes[0].grid(axis='x', alpha=0.3)

# CatBoost
cb_imp = pd.Series(cb_model.get_feature_importance(), index=TREE_FEAT).sort_values(ascending=False)
bar_colors_cb = ['crimson' if c in EMA_COLS else 'darkorange' for c in cb_imp.head(15).index]
cb_imp.head(15).plot(kind='barh', ax=axes[1], color=bar_colors_cb, alpha=0.85)
axes[1].invert_yaxis()
axes[1].set_title('CatBoost Top-15 Feature Importance\n(빨강=EMA 피처)', fontsize=12)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('v4.1.0 피처 중요도 — EMA 피처 기여도 확인', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./results/feature_importance_v4.1.0.png', dpi=150, bbox_inches='tight')
plt.show()

# EMA 피처 순위 출력
print('\n[LightGBM] EMA 피처 순위:')
lgb_all = lgb_imp.reset_index()
lgb_all.columns = ['feature', 'importance']
lgb_all['rank'] = range(1, len(lgb_all)+1)
print(lgb_all[lgb_all['feature'].isin(EMA_COLS)][['rank','feature','importance']].to_string(index=False))

print('\n[CatBoost] EMA 피처 순위:')
cb_all = cb_imp.reset_index()
cb_all.columns = ['feature', 'importance']
cb_all['rank'] = range(1, len(cb_all)+1)
print(cb_all[cb_all['feature'].isin(EMA_COLS)][['rank','feature','importance']].to_string(index=False))

---
## 10. 테스트 예측 + 후처리 + 제출

In [ ]:
lgb_test_trf = lgb_model.predict(X_lgb_test)
cb_test_trf  = cb_model.predict(X_cb_test)

is_tg_test = (test_pre['item'] == 'TG').values
lgb_test_pred = inverse_transform(lgb_test_trf, is_tg_test)
cb_test_pred  = inverse_transform(cb_test_trf,  is_tg_test)

if FINAL_MODE == 'ensemble':
    final_test_pred = (lgb_test_pred + cb_test_pred) / 2
elif FINAL_MODE == 'catboost':
    final_test_pred = cb_test_pred
else:
    final_test_pred = lgb_test_pred

final_test_pred = np.clip(final_test_pred, 0, None)

result_df = test_pre[['ID', 'item']].copy()
result_df['answer'] = final_test_pred

min_thresholds = {'TG': 400, 'CB': 50, 'RD': 10, 'CR': 150, 'BC': 100}
for item, thr in min_thresholds.items():
    mask = (result_df['item'] == item) & (result_df['answer'] < thr)
    result_df.loc[mask, 'answer'] = 0.0
    if mask.sum() > 0:
        print(f'{item}: {mask.sum()}개 → 0 처리')

print('\n예측 통계:')
print(result_df.groupby('item')['answer'].agg(['mean','min','max']).round(1))

In [ ]:
result = sub[['ID']].merge(result_df[['ID', 'answer']], on='ID', how='left')
result['answer'] = result['answer'].fillna(0.0)

SUBMISSION_PATH = './results/submission_v4.1.0.csv'
result.to_csv(SUBMISSION_PATH, index=False, encoding='utf-8-sig')
print(f'저장: {SUBMISSION_PATH}, 행: {len(result)}')
result.head(10)

---
## 11. 결과 요약

In [ ]:
print('=' * 65)
print('  v4.1.0 최종 결과')
print('=' * 65)
print(f'  [모델 구성]  CatBoost + LightGBM  (n_jobs={N_JOBS})')
print(f'  [신규 피처]  EMA 7/30/60/120일 (품목별 일평균, shift(1))')
print(f'  [검증 방법]  2022년 3월 holdout (테스트와 동일 계절)')
print(f'  [최종 선택]  {FINAL_MODE.upper()}')
print()
print('  [개별 모델 검증 MAE (2022년 3월)]')
print(f'    LightGBM  : {mae_lgb:>10,.2f} 원/kg  (best_iter={lgb_model.best_iteration_})')
print(f'    CatBoost  : {mae_cb:>10,.2f} 원/kg  (best_iter={cb_model.get_best_iteration()})')
print(f'    앙상블(avg): {mae_ens:>10,.2f} 원/kg')
print()
print('  [최종 검증 성능]')
print(f'    MAE  = {mae:>10,.2f} 원/kg')
print(f'    RMSE = {rmse:>10,.2f} 원/kg')
print(f'    R²   = {r2:>10.4f}')
print()
print(f'  제출 파일: {SUBMISSION_PATH}')
print('=' * 65)

### 다음 버전

| 버전 | 개선 내용 |
|------|----------|
| **v4.2.0** | 명절 접근 거리 피처 추가 (dist_to_seollal, dist_to_chuseok) — 박효준님 제안 |
| **v4.3.0** | 품목별 분리 모델 (TG / CR+RD / CB+BC) — 김대원님 제안 |
| **v4.4.0** | 무역 데이터 수출입 중량 피처 연동 — 박효준님 제안 |
| **v5.0.0** | Optuna 하이퍼파라미터 자동 최적화 (2022-03 검증 기준) |